In [ ]:
import os, re, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoTokenizer,
    ModernBertForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import matthews_corrcoef

RUNS_ROOTS = [
    Path("/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/runs/NT_benchmark/H3K4me3/H3K4me3_large_lr5e5_wd1e4_bs32_p50").resolve(),

]

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

for p in RUNS_ROOTS:
    assert p.exists(), f"runs_root not found: {p}"
print("runs:", len(RUNS_ROOTS))

_fold_re = re.compile(r"fold_(\d+)")

def fold_num(p: Path) -> int:
    m = _fold_re.search(p.name)
    return int(m.group(1)) if m else 10**9

def find_any_config_json(runs_root: Path):
    candidates = [
        runs_root / "config.json",
        *sorted(runs_root.glob("fold_*/config.json")),
        # *sorted(runs_root.glob("fold_*/*/config.json")),
    ]
    for c in candidates:
        if c.exists():
            with open(c, "r") as f:
                return json.load(f), c
    return None, None

def list_fold_dirs(runs_root: Path):
    # fold_0, fold_1, ... (and also fold_0_seed_42 style)
    dirs = [p for p in runs_root.iterdir() if p.is_dir() and _fold_re.search(p.name)]
    dirs = sorted(dirs, key=fold_num)
    return dirs

dataset_all = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks_revised")


def get_tokenized_test_dataset(task_name: str, tok_name: str, max_len: int):
    key = (task_name, tok_name, int(max_len))

    tokenizer = AutoTokenizer.from_pretrained(tok_name)

    def tokenization(batch):
        tok = tokenizer(
            batch["sequence"],
            add_special_tokens=True,
            truncation=True,
            max_length=max_len,
            padding=False,  # <-- important: NO padding here (like training)
        )
        tok["labels"] = batch["label"]
        return tok

    ds = dataset_all["test"].filter(lambda ex: ex["task"] == task_name)
    ds = ds.map(tokenization, batched=True, remove_columns=ds.column_names)


    return ds

def make_test_loader(task_name: str, tok_name: str, max_len: int, batch_size: int, num_workers: int):
    tokenizer = AutoTokenizer.from_pretrained(tok_name)
    ds = get_tokenized_test_dataset(task_name, tok_name, max_len)

    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding="longest",      # <-- pad to max length IN THE BATCH
        return_tensors="pt",
        pad_to_multiple_of=8,
    )

    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        collate_fn=collator,
    )
    return loader, len(ds)


def load_model_from_model_best(model_cfg, model_best_dir: Path):
    ckpt = model_best_dir / "pytorch_model.bin"
    if not ckpt.exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt}")

    model = ModernBertForSequenceClassification(config=model_cfg)

    sd = torch.load(ckpt, map_location="cpu")
    sd = {k.replace("module.", ""): v for k, v in sd.items()}

    missing, unexpected = model.load_state_dict(sd, strict=True)
    if missing or unexpected:
        raise RuntimeError(
            f"State dict mismatch in {model_best_dir}. missing={len(missing)}, unexpected={len(unexpected)}"
        )

    model.to(DEVICE)
    model.eval()
    return model

@torch.no_grad()
def eval_mcc(model, test_loader):
    ys, ps = [], []
    for batch in test_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if torch.is_tensor(v)}

        tti = batch.get("token_type_ids", None)
        if tti is None:
            tti = torch.zeros_like(batch["input_ids"])

        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=tti,
        ).logits

        preds = torch.argmax(logits, dim=1)
        ys.append(batch["labels"].detach().cpu().numpy())
        ps.append(preds.detach().cpu().numpy())

    y = np.concatenate(ys)
    p = np.concatenate(ps)
    return float(matthews_corrcoef(y, p))

records = []

for runs_root in RUNS_ROOTS:
    config_folder_name = runs_root.parent.name  
    run_name = runs_root.name

    run_cfg, cfg_path = find_any_config_json(runs_root)
    if run_cfg is None:
        raise FileNotFoundError(
            f"No config.json found in {runs_root} (or fold_*/config.json). "
            "Check your run directory structure."
        )

    task_name   = run_cfg.get("task_name", None) or runs_root.parent.name
    tok_name    = run_cfg.get("tokenizer", None)
    max_len     = int(run_cfg.get("input_seq_len", 1024))
    cls_num     = int(run_cfg.get("cls_num", 2))
    hf_model    = run_cfg.get("hf_model_name", None)
    attn_impl   = run_cfg.get("attn_implementation", None)

    batch_size  = int(run_cfg.get("batch_size", 32))
    num_workers = int(run_cfg.get("data_n_workers", 2))

    if tok_name is None:
        raise KeyError(f"'tokenizer' not found in config.json at {cfg_path}")
    if hf_model is None:
        raise KeyError(f"'hf_model_name' not found in config.json at {cfg_path}")

    print(f"\nRUN: {runs_root}")
    print(f"  config.json: {cfg_path}")
    print(f"  task={task_name} | tokenizer={tok_name} | max_len={max_len} | cls_num={cls_num}")
    print(f"  hf_model={hf_model} | attn_impl={attn_impl} | bs_eval={batch_size} | nw={num_workers}")

    test_loader, test_n = make_test_loader(task_name, tok_name, max_len, batch_size, num_workers)
    print(f"  test_n={test_n}")

    model_cfg = AutoConfig.from_pretrained(hf_model, trust_remote_code=True)
    model_cfg.num_labels = cls_num
    model_cfg.problem_type = "single_label_classification"
    if attn_impl is not None:
        model_cfg.attn_implementation = attn_impl

    fold_dirs = list_fold_dirs(runs_root)
    if not fold_dirs:
        raise FileNotFoundError(f"No fold_* directories found in {runs_root}")

    rec = {
        "config": config_folder_name,
        "run": run_name,
        "path": str(runs_root),
        "task_name": task_name,
        "tokenizer": tok_name,
        "max_len": max_len,
        "cls_num": cls_num,
        "hf_model_name": hf_model,
        "attn_impl": attn_impl,
        "batch_size_eval": batch_size,
        "test_n": test_n,
        "n_folds": len(fold_dirs),
        "config_json_path": str(cfg_path),
    }

    mcc_by_fold = {}
    errors = []

    for fd in fold_dirs:
        f = fold_num(fd)
        try:
            model_best = fd / "model_best"
            model = load_model_from_model_best(model_cfg, model_best)
            mcc = eval_mcc(model, test_loader)
            mcc_by_fold[f] = mcc
            print(f"  {fd.name}: mcc={mcc:.6f}")
        except Exception as e:
            mcc_by_fold[f] = np.nan
            errors.append(f"[{fd.name}] {type(e).__name__}: {e}")
            print(f"  {fd.name}: ERROR -> {type(e).__name__}: {e}")

    ordered_folds = sorted(mcc_by_fold.keys())
    mcc_list = [mcc_by_fold[f] for f in ordered_folds]
    rec["mccs"] = mcc_list

    for f in ordered_folds:
        rec[f"mcc_fold_{f}"] = mcc_by_fold[f]

    arr = np.array(mcc_list, dtype=float)
    finite = arr[np.isfinite(arr)]
    rec["mean_mcc"] = float(np.mean(finite)) if finite.size else np.nan
    rec["std_mcc"]  = float(np.std(finite, ddof=1)) if finite.size >= 2 else (0.0 if finite.size == 1 else np.nan)

    if errors:
        rec["error"] = "\n".join(errors)

    records.append(rec)

df = pd.DataFrame(records)

fold_cols = sorted([c for c in df.columns if c.startswith("mcc_fold_")],
                   key=lambda x: int(x.split("_")[-1]))
front = [
    "config", "run", "task_name",
    "n_folds", "test_n",
    "mean_mcc", "std_mcc", "mccs",
    "tokenizer", "max_len", "cls_num",
    "hf_model_name", "attn_impl",
    "batch_size_eval",
    "config_json_path",
    "path",
]

front = [c for c in front if c in df.columns]
rest = [c for c in df.columns if c not in front + fold_cols]
df = df[front + fold_cols + rest].sort_values(["config", "run"]).reset_index(drop=True)
df.to_csv('/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/nucleotide_transformer_bench/results.csv', index = False)
display(df)



/home/jovyan/miniconda3/envs/expression/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE: cuda:0
runs: 1

RUN: /home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/runs/NT_benchmark/H3K4me3/H3K4me3_large_lr5e5_wd1e4_bs32_p50
  config.json: /home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/runs/NT_benchmark/H3K4me3/H3K4me3_large_lr5e5_wd1e4_bs32_p50/fold_0/config.json
  task=H3K4me3 | tokenizer=AIRI-Institute/gena-lm-bert-base-t2t | max_len=1024 | cls_num=2
  hf_model=/home/jovyan/shares/SR003.nfs2/aspeedok/models/modernbert_large | attn_impl=sdpa | bs_eval=32 | nw=2


Map: 100%|██████████| 776/776 [00:00<00:00, 2778.37 examples/s]


  test_n=776
  fold_0: mcc=0.636767
  fold_1: mcc=0.642523
  fold_2: mcc=0.638929
  fold_3: mcc=0.663694
  fold_4: mcc=0.621186
  fold_5: mcc=0.645781
  fold_6: mcc=0.632556
  fold_7: mcc=0.649700
  fold_8: mcc=0.642369
  fold_9: mcc=0.657219


,config,run,task_name,n_folds,test_n,mean_mcc,std_mcc,mccs,tokenizer,max_len,...,mcc_fold_0,mcc_fold_1,mcc_fold_2,mcc_fold_3,mcc_fold_4,mcc_fold_5,mcc_fold_6,mcc_fold_7,mcc_fold_8,mcc_fold_9
0,H3K4me3,H3K4me3_large_lr5e5_wd1e4_bs32_p50,H3K4me3,10,776,0.643072,0.012129,"[0.6367674380449851, 0.6425234156554981, 0.638...",AIRI-Institute/gena-lm-bert-base-t2t,1024,...,0.636767,0.642523,0.638929,0.663694,0.621186,0.645781,0.632556,0.6497,0.642369,0.657219


In [11]:
df.to_csv('/home/jovyan/shares/SR003.nfs2/aspeedok/GENA_LM/downstream_tasks/nucleotide_transformer_bench/results.csv', index = False)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True,)
model = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

/home/jovyan/miniconda3/envs/dna310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jovyan/miniconda3/envs/dna310/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNABERT-2-117M:
- flash_attn_triton.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNABERT-2-117M:
- bert_padding.py
. Make sure to do

In [4]:
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    trust_remote_code=True,
    revision="ec1f874253852eb3907081f57294991b4280ceb6",
    torch_dtype=torch.bfloat16,   # или torch.float16
).cuda().eval()


Some weights of the model checkpoint at zhihan1996/DNABERT-2-117M were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should prob

In [5]:
import sys

print("cuda:", next(model.parameters()).is_cuda)
print("dtype:", next(model.parameters()).dtype)
print("attn dropout in config:", model.config.attention_probs_dropout_prob)

# Найдём модуль remote-code bert_layers, где лежит flash_attn_qkvpacked_func
bert_layers_mod = None
for m in sys.modules.values():
    if hasattr(m, "flash_attn_qkvpacked_func") and hasattr(m, "BertUnpadSelfAttention"):
        bert_layers_mod = m
        break

print("bert_layers module:", None if bert_layers_mod is None else bert_layers_mod.__name__)
print("flash_attn_qkvpacked_func is None?:", None if bert_layers_mod is None else (bert_layers_mod.flash_attn_qkvpacked_func is None))


cuda: True
dtype: torch.bfloat16
attn dropout in config: 0.0
bert_layers module: transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.bert_layers
flash_attn_qkvpacked_func is None?: False


In [6]:
import sys
import torch


bert_layers_mod = None
for m in sys.modules.values():
    if hasattr(m, "flash_attn_qkvpacked_func") and hasattr(m, "BertUnpadSelfAttention"):
        bert_layers_mod = m
        break

assert bert_layers_mod is not None, "Не нашёл bert_layers модуль"
assert bert_layers_mod.flash_attn_qkvpacked_func is not None, "flash_attn_qkvpacked_func == None (будет fallback)"


called = {"n": 0}
orig = bert_layers_mod.flash_attn_qkvpacked_func

def wrapped(*args, **kwargs):
    called["n"] += 1
    return orig(*args, **kwargs)

bert_layers_mod.flash_attn_qkvpacked_func = wrapped

dna = "ACGT" * 256
inputs = tokenizer(dna, return_tensors="pt").to("cuda")

with torch.no_grad():
    _ = model(**inputs)

print("FlashAttention calls:", called["n"])


CompilationError: at 114:24:        else:
            if EVEN_HEADDIM:
                k = tl.load(k_ptrs + start_n * stride_kn,
                            mask=(start_n + offs_n)[:, None] < seqlen_k,
                            other=0.0)
            else:
                k = tl.load(k_ptrs + start_n * stride_kn,
                            mask=((start_n + offs_n)[:, None] < seqlen_k) &
                            (offs_d[None, :] < headdim),
                            other=0.0)
        qk = tl.zeros([BLOCK_M, BLOCK_N], dtype=tl.float32)
        qk += tl.dot(q, k, trans_b=True)
                        ^
TypeError("dot() got an unexpected keyword argument 'trans_b'")

In [16]:
import torch
from transformers import AutoTokenizer, AutoModel

REV = "ec1f874253852eb3907081f57294991b4280ceb6" 

tokenizer = AutoTokenizer.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    trust_remote_code=True,
    revision=REV,
    force_download=True,
)

model = AutoModel.from_pretrained(
    "zhihan1996/DNABERT-2-117M",
    trust_remote_code=True,
    revision=REV,
    force_download=True,
    torch_dtype=torch.bfloat16,
).cuda().eval()


/home/jovyan/miniconda3/envs/dna310/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNABERT-2-117M:
- flash_attn_triton.py
- bert_padding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Some weights of the model checkpoint at zhihan1996/DNABERT-2-117M were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing Ber

In [18]:
import sys

fa = None
for m in sys.modules.values():
    f = getattr(m, "__file__", None)
    if isinstance(f, str) and f.endswith("flash_attn_triton.py"):
        fa = m
        break

print("fa module:", None if fa is None else fa.__name__)
print("flash_attn_triton.py path:", None if fa is None else fa.__file__)


fa module: transformers_modules.zhihan1996.DNABERT-2-117M.7bce263b15377fc15361f52cfab88f8b586abda0.flash_attn_triton
flash_attn_triton.py path: /home/jovyan/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/flash_attn_triton.py


In [19]:
txt = open(fa.__file__, "r", encoding="utf-8").read()
print("contains trans_b=True ?", "trans_b=True" in txt)


contains trans_b=True ? True


In [13]:
import sys
fa = None
for m in sys.modules.values():
    f = getattr(m, "__file__", None)
    if isinstance(f, str) and f.endswith("flash_attn_triton.py"):
        fa = m
        break

txt = open(fa.__file__, "r", encoding="utf-8").read()
print("contains trans_b=True ?", "trans_b=True" in txt)


contains trans_b=True ? False


In [12]:
import re, pathlib

path = pathlib.Path("/home/jovyan/.cache/huggingface/modules/transformers_modules/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/flash_attn_triton.py")
txt = path.read_text(encoding="utf-8")

txt2 = re.sub(
    r"tl\.dot\(\s*([^,]+)\s*,\s*([^,]+)\s*,\s*trans_b=True\s*\)",
    r"tl.dot(\1, tl.trans(\2))",
    txt
)

path.write_text(txt2, encoding="utf-8")
print("patched, trans_b=True left?", "trans_b=True" in txt2)


patched, trans_b=True left? False


In [14]:
import sys
import torch

# найти bert_layers модуль
bert_layers_mod = None
for m in sys.modules.values():
    if hasattr(m, "flash_attn_qkvpacked_func") and hasattr(m, "BertUnpadSelfAttention"):
        bert_layers_mod = m
        break

print("flash_attn_qkvpacked_func is None?:", bert_layers_mod.flash_attn_qkvpacked_func is None)

called = {"n": 0}
orig = bert_layers_mod.flash_attn_qkvpacked_func

def wrapped(*args, **kwargs):
    called["n"] += 1
    return orig(*args, **kwargs)

bert_layers_mod.flash_attn_qkvpacked_func = wrapped

dna = "ACGT" * 256
inputs = tokenizer(dna, return_tensors="pt").to("cuda")

with torch.no_grad():
    _ = model(**inputs)

print("FlashAttention calls:", called["n"])


flash_attn_qkvpacked_func is None?: False


RecursionError: maximum recursion depth exceeded

In [ ]:
import sys
import torch

# 1) найдём модуль bert_layers (где используется flash_attn_qkvpacked_func)
bert_layers_mod = None
for m in sys.modules.values():
    if hasattr(m, "flash_attn_qkvpacked_func") and hasattr(m, "BertUnpadSelfAttention"):
        bert_layers_mod = m
        break
assert bert_layers_mod is not None, "Не нашёл bert_layers модуль"

# 2) найдём модуль flash_attn_triton (где определён оригинал)
fa_mod = None
for m in sys.modules.values():
    name = getattr(m, "__name__", "")
    if isinstance(name, str) and name.endswith(".flash_attn_triton"):
        fa_mod = m
        break
assert fa_mod is not None, "Не нашёл flash_attn_triton модуль"

# 3) если ранее уже оборачивали — вернём оригинал
cur = bert_layers_mod.flash_attn_qkvpacked_func
if hasattr(cur, "_flashattn_orig"):
    bert_layers_mod.flash_attn_qkvpacked_func = cur._flashattn_orig

# 4) оборачиваем ОРИГИНАЛ (берём из flash_attn_triton, а не из bert_layers)
orig = fa_mod.flash_attn_qkvpacked_func
called = {"n": 0}

def wrapped(*args, **kwargs):
    called["n"] += 1
    return orig(*args, **kwargs)

# маркеры, чтобы можно было “развернуть” потом
wrapped._flashattn_orig = orig

bert_layers_mod.flash_attn_qkvpacked_func = wrapped

# 5) прогон
dna = "ACGT" * 256
inputs = tokenizer(dna, return_tensors="pt").to("cuda")

with torch.no_grad():
    _ = model(**inputs)

print("FlashAttention calls:", called["n"])
